# Psychology RAFT-RAG — QLoRA Fine-Tuning
This notebook prepares a small Qwen model for psychology psychoeducation using QLoRA. Run on a Google Colab T4 GPU.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers==4.46.3 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 datasets==3.0.1 sentencepiece

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration
Set `DATA_PATH` to the psychology RAFT JSONL dataset produced by the RAFT preparation stage.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
DATA_PATH = '/content/psychology_raft_train.jsonl'
OUTPUT_DIR = '/content/psychology-qwen-qlora'
MAX_LENGTH = 1024

In [ ]:
from datasets import load_dataset
dataset = load_dataset('json', data_files=DATA_PATH, split='train')
print(dataset)
print(dataset.column_names)

## 3. Convert RAFT records to training text
The helper accepts common chat-style records and instruction/context/answer records. Inspect one sample before training.

In [ ]:
def record_to_text(row):
    if 'messages' in row:
        messages = row['messages']
        if isinstance(messages, str):
            return messages
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    instruction = row.get('instruction') or row.get('question') or ''
    context = row.get('context') or ''
    answer = row.get('answer') or row.get('completion') or row.get('cot_answer') or ''
    return f'<|im_start|>system\\nYou are a psychology psychoeducation assistant. Use the provided context and do not diagnose.\\n<|im_end|>\\n<|im_start|>user\\nContext:\\n{context}\\n\\nQuestion: {instruction}\\n<|im_end|>\\n<|im_start|>assistant\\n{answer}\\n<|im_end|>'


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(record_to_text(dataset[0])[:2000])

In [ ]:
def tokenize(row):
    text = record_to_text(row)
    return tokenizer(text, truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
print(tokenized)

## 4. Load Qwen in 4-bit mode

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.config.use_cache = False

## 5. Add the QLoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train
Start with a small number of epochs for the student project. Increase only after checking validation quality and GPU memory.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
training_args = TrainingArguments(output_dir=OUTPUT_DIR, num_train_epochs=1, per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4, logging_steps=10, save_strategy='steps', save_steps=100, save_total_limit=2, fp16=(compute_dtype==torch.float16), bf16=(compute_dtype==torch.bfloat16), report_to='none', optim='paged_adamw_8bit')
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized, data_collator=collator)
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter/model files to:', OUTPUT_DIR)

## 7. Important checks
Before using the adapter in RAG inference, test several questions and verify that responses remain grounded, non-diagnostic, and safe.